+ 심리학/인지언어학에서 점화효과.
+ 딥러닝 계층을 깊게 쌓는 것과 비슷하다.
+ 깊게 쌓을 수록 더 추상적인 생각. 그렇다면 얼마나 깊게 쌓아야하는가?
+ 너무 깊게 쌓으면 사람이 딴 생각을 하듯, 딥러닝 네트워크 역시 다른 생각으로 빠질 수도 있을 것 같다.

<br>

> 문맥의 문제!! 단어의 모호성/중의성 등을 이미지에 기반한 priming effect로 해결할 수 있지 않을까???

In [ ]:
import os
os.sys.path.append('../../official_github/')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from common.multi_layer_net_extend import MultiLayerNetExtend
from common.optimizer import SGD, Adam

In [ ]:
test_dict = {}
train_dict = {}
root_path = './cifar-10-batches-py/'


def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

accepted_keys = [b'labels', b'data'] ## select keys from [b'batch_label', b'labels', b'data', b'filenames']
for dir in os.listdir(root_path):
    if dir == 'batches.meta':
        pass
    elif dir == 'test_batch':
        test_batch = unpickle(os.path.join(root_path, dir))
        test_dict.update((k,np.array(v)) for k,v in test_batch.items() if k in accepted_keys)
    else:
        train_batch = unpickle(os.path.join(root_path, dir))
        if not train_dict:
            train_dict.update((k,np.array(v)) for k,v in train_batch.items() if k in accepted_keys)
        else:
            for key in accepted_keys:
                train_dict[key] = np.concatenate((train_dict[key], train_batch[key]), axis=0)


## CIFAR data's shape are (N, H*W*C). 
## If you want to convert them into numpy-standard (N, H, W, C) shape, run following codes.
def numpy2rgb(arr: np.ndarray) -> np.ndarray:
    batch_size = len(arr)
    arr = arr.reshape(batch_size, 3, 32, 32).transpose(0,2,3,1)
    
    return arr


train_dict[b'data'] = numpy2rgb(train_dict[b'data'])
test_dict[b'data'] = numpy2rgb(test_dict[b'data'])

In [ ]:
x_train, t_train = train_dict[b'data'], train_dict[b'labels']
x_test, t_test = test_dict[b'data'], test_dict[b'labels']

In [ ]:
max_epochs = 20
train_size = x_train.shape[0]
batch_size = 100

iters = 5_000
iter_per_epoch = max(train_size / batch_size, 1)

learning_rate = 0.01

network_dict = {'Affine_3': MultiLayerNetExtend(input_size=3072, hidden_size_list=[1000, 100], output_size=10, weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                'Affine_4': MultiLayerNetExtend(input_size=3072, hidden_size_list=[1000, 100, 50], output_size=10, weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                }

output_dict = {}

for key, network in network_dict.items():
    net = network
    optimizer = Adam(lr=learning_rate)

    output_dict[f'{key} train acc'] = []
    output_dict[f'{key} test acc'] = []
    
    print(f'=== {key} network training start ===')
    for i in tqdm(range(iters), leave=False):
        batch_mask = np.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        t_batch = t_train[batch_mask]

        grads = net.gradient(x_batch, t_batch)
        optimizer.update(net.params, grads)

        if i % iter_per_epoch == 0:
            train_acc = net.accuracy(x_train, t_train)
            test_acc = net.accuracy(x_test, t_test)
            output_dict[f'{key} train acc'].append(train_acc)
            output_dict[f'{key} test acc'].append(test_acc)
            print(f'=== {key} result report ===')
            print(f'train_acc: {train_acc}, test_acc: {test_acc}')
    
    

In [13]:
network_dict['Affine_2'].params

{'W1': array([[ 4.06472962e-03, -1.17221131e-02,  4.07205828e-05, ...,
          4.16084904e-05, -1.38194218e-03,  1.98182451e-05],
        [ 5.67344896e-04, -1.00765458e-02, -1.94658694e-05, ...,
          1.18064998e-05, -8.19137559e-04, -4.79748857e-06],
        [-4.60243416e-03, -2.90396900e-03,  1.12639854e-05, ...,
         -5.35506853e-07, -2.45210029e-03,  2.32804760e-05],
        ...,
        [ 6.25237849e-03, -8.59479069e-03, -1.79727802e-05, ...,
         -7.72005007e-05, -1.29671070e-03, -5.59865407e-06],
        [ 4.79304006e-03, -9.29109082e-03,  1.81958530e-05, ...,
         -4.13826515e-05, -8.59917671e-04,  8.89353572e-06],
        [ 3.12644271e-03, -7.37450003e-03, -2.08126424e-05, ...,
         -1.20383508e-04, -5.75629130e-04,  8.89512097e-07]]),
 'b1': array([-8.96479369e-15,  2.36081379e-14, -2.92569054e-15, -6.11745349e-16,
         8.51118581e-15,  8.56633166e-15,  1.11545637e-15, -6.69992807e-17,
         5.26592552e-15,  5.92270339e-15,  6.11164027e-16,  1.257

In [16]:
import pickle

with open('affine_4.pkl', 'wb') as f:
    pickle.dump(network_dict['Affine_4'].params, f)
    

### hyper parameters
  + max_epochs = 20
  + batch_size = 100
  + iters = 5_000
  + learning_rate = 0.01

<br>

### results
+ **before image transposing** Affine_2([1000]): train_acc: 0.16316, test_acc: 0.1612, running_time: 20m
+ Affine_2[1000]: train_acc: 0.12014, test_acc: 0.1223
+ Affine_3[1000, 100]: train_acc: 0.18088, test_acc: 0.1842
+ Affine_3[1000, 500]: train_acc: 0.10366, test_acc: 0.1038
+ Affine_4[1000, 100, 50]: train_acc: 0.15788, test_acc: 0.1586
+ Affine_4[100, 100, 100]: train_acc: 0.1025, test_acc: 0.103
<br>

+ 